# SalesLens - Data Exploration

## 1. Project Objective


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.raw_loader import list_raw_csv_files, load_raw_csv

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


## 2. Load Raw Data

In [ ]:
files = list_raw_csv_files()
files

[WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_customers_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_geolocation_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_order_items_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_order_payments_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_order_reviews_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_orders_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_products_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/olist_sellers_dataset.csv'),
 WindowsPath('D:/PROJETS/SalesLens/data/raw/product_category_name_translation.csv')]

## 3. Dataset Overview

In [ ]:
data = {p.name: load_raw_csv(p.name) for p in files}
overview = pd.DataFrame([
    {
        'Dataset': name,
        'Nombre de lignes': len(df),
        'Nombre de colonnes': len(df.columns),
        'Valeurs manquantes': int(df.isna().sum().sum()),
        'Doublons': int(df.duplicated().sum()),
    }
    for name, df in data.items()
]).sort_values('Dataset')
overview

,Dataset,Nombre de lignes,Nombre de colonnes,Valeurs manquantes,Doublons
0,olist_customers_dataset.csv,99441,5,0,0
1,olist_geolocation_dataset.csv,1000163,5,0,261831
2,olist_order_items_dataset.csv,112650,7,0,0
3,olist_order_payments_dataset.csv,103886,5,0,0
4,olist_order_reviews_dataset.csv,99224,7,145903,0
5,olist_orders_dataset.csv,99441,8,4908,0
6,olist_products_dataset.csv,32951,9,2448,0
7,olist_sellers_dataset.csv,3095,4,0,0
8,product_category_name_translation.csv,71,2,0,0


## 4. Missing Values

In [ ]:
for name, df in data.items():
    print(f'\n### {name}')
    print(df.isna().sum().sort_values(ascending=False))



### olist_customers_dataset.csv
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

### olist_geolocation_dataset.csv
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

### olist_order_items_dataset.csv
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

### olist_order_payments_dataset.csv


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

### olist_order_reviews_dataset.csv
review_comment_title       87656
review_comment_message     58247
review_id                      0
review_score                   0
order_id                       0
review_creation_date           0
review_answer_timestamp        0
dtype: int64

### olist_orders_dataset.csv
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

### olist_products_dataset.csv
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm          

## 5. Duplicate Records

In [ ]:
for name, df in data.items():
    print(f'{name}: {df.duplicated().sum()} duplicates')


olist_customers_dataset.csv: 0 duplicates


olist_geolocation_dataset.csv: 261831 duplicates
olist_order_items_dataset.csv: 0 duplicates


olist_order_payments_dataset.csv: 0 duplicates


olist_order_reviews_dataset.csv: 0 duplicates


olist_orders_dataset.csv: 0 duplicates
olist_products_dataset.csv: 0 duplicates
olist_sellers_dataset.csv: 0 duplicates
product_category_name_translation.csv: 0 duplicates


## 6. Identifiers and Relationships

In [ ]:
for name, df in data.items():
    print(f'\n### {name}')
    id_cols = [c for c in df.columns if c.endswith('_id') or c.endswith('_code_prefix') or c in {'review_id', 'order_item_id', 'payment_sequential'}]
    for c in id_cols:
        print(f'{c}: unique={df[c].nunique(dropna=False)} / rows={len(df)}')



### olist_customers_dataset.csv
customer_id: unique=99441 / rows=99441
customer_unique_id: unique=96096 / rows=99441
customer_zip_code_prefix: unique=14994 / rows=99441

### olist_geolocation_dataset.csv
geolocation_zip_code_prefix: unique=19015 / rows=1000163

### olist_order_items_dataset.csv


order_id: unique=98666 / rows=112650
order_item_id: unique=21 / rows=112650
product_id: unique=32951 / rows=112650


seller_id: unique=3095 / rows=112650

### olist_order_payments_dataset.csv
order_id: unique=99440 / rows=103886
payment_sequential: unique=29 / rows=103886

### olist_order_reviews_dataset.csv
review_id: unique=98410 / rows=99224


order_id: unique=98673 / rows=99224

### olist_orders_dataset.csv


order_id: unique=99441 / rows=99441
customer_id: unique=99441 / rows=99441

### olist_products_dataset.csv
product_id: unique=32951 / rows=32951

### olist_sellers_dataset.csv
seller_id: unique=3095 / rows=3095
seller_zip_code_prefix: unique=2246 / rows=3095

### product_category_name_translation.csv


## 7. Date Columns

In [ ]:
date_candidates = {
    'olist_order_items_dataset.csv': ['shipping_limit_date'],
    'olist_order_reviews_dataset.csv': ['review_creation_date', 'review_answer_timestamp'],
    'olist_orders_dataset.csv': [
        'order_purchase_timestamp', 'order_approved_at',
        'order_delivered_carrier_date', 'order_delivered_customer_date',
        'order_estimated_delivery_date']
}
for name, cols in date_candidates.items():
    df = data[name]
    print(f'\n### {name}')
    for c in cols:
        parsed = pd.to_datetime(df[c], errors='coerce')
        print(c, '| dtype=', df[c].dtype, '| parsed_dtype=', parsed.dtype, '| min=', parsed.min(), '| max=', parsed.max())



### olist_order_items_dataset.csv


shipping_limit_date | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-09-19 00:15:34 | max= 2020-04-09 22:35:08

### olist_order_reviews_dataset.csv
review_creation_date | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-10-02 00:00:00 | max= 2018-08-31 00:00:00


review_answer_timestamp | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-10-07 18:32:28 | max= 2018-10-29 12:27:35

### olist_orders_dataset.csv


order_purchase_timestamp | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-09-04 21:15:19 | max= 2018-10-17 17:30:18


order_approved_at | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-09-15 12:16:38 | max= 2018-09-03 17:40:06


order_delivered_carrier_date | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-10-08 10:34:01 | max= 2018-09-11 19:48:28
order_delivered_customer_date | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-10-11 13:46:32 | max= 2018-10-17 13:22:46
order_estimated_delivery_date | dtype= object | parsed_dtype= datetime64[ns] | min= 2016-09-30 00:00:00 | max= 2018-11-12 00:00:00


## 8. Numerical Variables

In [ ]:
numeric_cols = {
    'olist_order_items_dataset.csv': ['price', 'freight_value'],
    'olist_order_payments_dataset.csv': ['payment_installments', 'payment_value'],
    'olist_order_reviews_dataset.csv': ['review_score'],
    'olist_products_dataset.csv': ['product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm'],
}
for name, cols in numeric_cols.items():
    df = data[name]
    print(f'\n### {name}')
    print(df[cols].describe().T[['count', 'mean', 'min', '50%', 'max']])



### olist_order_items_dataset.csv
                  count        mean   min    50%      max
price          112650.0  120.653739  0.85  74.99  6735.00
freight_value  112650.0   19.990320  0.00  16.26   409.68

### olist_order_payments_dataset.csv
                         count        mean  min    50%       max
payment_installments  103886.0    2.853349  0.0    1.0     24.00
payment_value         103886.0  154.100380  0.0  100.0  13664.08

### olist_order_reviews_dataset.csv
                count      mean  min  50%  max
review_score  99224.0  4.086421  1.0  5.0  5.0

### olist_products_dataset.csv


                              count         mean  min    50%      max
product_name_lenght         32341.0    48.476949  5.0   51.0     76.0
product_description_lenght  32341.0   771.495285  4.0  595.0   3992.0
product_photos_qty          32341.0     2.188986  1.0    1.0     20.0
product_weight_g            32949.0  2276.472488  0.0  700.0  40425.0
product_length_cm           32949.0    30.815078  7.0   25.0    105.0
product_height_cm           32949.0    16.937661  2.0   13.0    105.0
product_width_cm            32949.0    23.196728  6.0   20.0    118.0


## 9. Categorical Variables

In [ ]:
cat_cols = {
    'olist_orders_dataset.csv': ['order_status'],
    'olist_order_payments_dataset.csv': ['payment_type'],
    'olist_customers_dataset.csv': ['customer_state'],
    'olist_sellers_dataset.csv': ['seller_state'],
    'olist_products_dataset.csv': ['product_category_name'],
}
for name, cols in cat_cols.items():
    df = data[name]
    print(f'\n### {name}')
    for c in cols:
        print(f'\n{c}')
        print(df[c].value_counts(dropna=False).head(10))



### olist_orders_dataset.csv

order_status
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### olist_order_payments_dataset.csv

payment_type
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

### olist_customers_dataset.csv

customer_state
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

### olist_sellers_dataset.csv

seller_state
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
RS     129
GO      40
DF      30
ES      23
BA      19
Name: count, dtype: int64

### olist_products_dataset.csv

product_category_name
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_dec